In [63]:
import pandas as pd
import numpy as np
import re

In [64]:
#pip install xlrd

In [65]:
#Download the file
df = pd.read_excel("GSAF5.xls")
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,...,Species,Source,pdf,href formula,href,Case Number,Case Number.1,original order,Unnamed: 21,Unnamed: 22
0,10th January,2026.0,Unprovoked,Australia,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,...,Unknown,Bob Myatt GSAF,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8th January,2026.0,Unprovoked,US Virgin Islands,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,...,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3rd January,2026.0,Unprovoked,New Caledonia,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,...,Unknown,Andy Currie: Province Sud:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,...,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,...,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
#Eliminar columnas por nombre 
columns_to_drop = { 
    'pdf',
    'href',
    'href formula',
    'Case Number',
    'Case Number.1',
    'original order',
    'Unnamed: 21',
    'Unnamed: 22'
}
df = df.drop(columns=columns_to_drop)

In [67]:
df.shape

(7065, 15)

In [68]:
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source
0,10th January,2026.0,Unprovoked,Australia,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,puncture mark to left thumb,N,0540hrs,Unknown,Bob Myatt GSAF
1,8th January,2026.0,Unprovoked,US Virgin Islands,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,Left arm torn off in the attack below the elbow,Y,1628hrs,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com
2,3rd January,2026.0,Unprovoked,New Caledonia,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,Injuries to upper limbs,N,?,Unknown,Andy Currie: Province Sud:
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,Taken by shark body recovered with multiple in...,Y,1200hrs,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,Hand Injury,N,0800hrs,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...


#### ver los valores nulos y duplicados

In [69]:
df.isna().sum()/df.shape[0]

Date         0.000000
Year         0.000283
Type         0.002548
Country      0.007077
State        0.068931
Location     0.080255
Activity     0.082803
Name         0.030998
Sex          0.081953
Age          0.423921
Injury       0.004954
Fatal Y/N    0.079406
Time         0.499222
Species      0.443171
Source       0.002831
dtype: float64

In [70]:
df.duplicated().sum()

1

In [71]:
filas_duplicadas = df[df.duplicated(keep=False)]
filas_duplicadas

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source
5436,Fall 1943,1943.0,Unprovoked,USA,Hawaii,"Midway Island, Northwestern Hawaiian Islands",Spearfishing,2 males,M,NaN,Calf nipped in each case,N,NaN,"""small sharks""",W. M. Chapman
5437,Fall 1943,1943.0,Unprovoked,USA,Hawaii,"Midway Island, Northwestern Hawaiian Islands",Spearfishing,2 males,M,NaN,Calf nipped in each case,N,NaN,"""small sharks""",W. M. Chapman


In [72]:
#eliminando duplicados 
df = df.drop(5436)

In [73]:
df.shape

(7064, 15)

### Column Type

In [17]:
#Looks Type 
df['Type'].unique()

array(['Unprovoked', 'Provoked', 'Questionable', 'unprovoked',
       ' Provoked', 'Watercraft', 'Sea Disaster', nan, '?', 'Unconfirmed',
       'Unverified', 'Invalid', 'Under investigation', 'Boat'],
      dtype=object)

In [18]:
#Clean type
df['Type'] = df['Type'].astype(str).str.strip().str.capitalize()

#Unificar valores raros a 'Unkown
df['Type'] = df['Type'].replace({
    'Unverified': 'Unknown',
    'Unconfirmed': 'Unknown',
    '?': 'Unknown',
    'Nan': 'Unknown',         
    'Invalid': 'Unknown',
    'Under investigation': 'Unknown',
    'Questionable' : 'Unknown'
})

#unificar 'Unprovoked' y 'Provoked' (ya los capitalizamos)
df['Type'] = df['Type'].replace({'Unprovoked': 'Unprovoked', 'Provoked': 'Provoked'})

#otros tipos 
df['Type'] = df['Type'].replace({'Watercraft': 'Other', 'Sea disaster': 'Other', 'Boat': 'Other'})

In [19]:
#Los cambios se hicieron efectivos 
df['Type'].unique()

array(['Unprovoked', 'Provoked', 'Unknown', 'Other'], dtype=object)

### Column Country

In [20]:
df['Country'].unique()

array(['Australia', 'US Virgin Islands', 'New Caledonia', 'USA',
       'French Polynesia', 'Samoa', 'Columbia', 'Costa Rica', 'Bahamas',
       'Puerto Rico', 'Spain', 'Canary Islands', 'South Africa',
       'Vanuatu', 'Jamaica', 'Israel', 'Mexico', 'Maldives',
       'Philippines', 'Turks and Caicos', 'Mozambique', 'Egypt',
       'Thailand', 'New Zealand', 'Hawaii', 'Honduras', 'Indonesia',
       'Morocco', 'Belize', 'Maldive Islands', 'Tobago', 'AUSTRALIA',
       'INDIA', 'TRINIDAD', 'BAHAMAS', 'SOUTH AFRICA', 'MEXICO',
       'NEW ZEALAND', 'EGYPT', 'BELIZE', 'PHILIPPINES', 'Coral Sea',
       'SPAIN', 'PORTUGAL', 'SAMOA', 'COLOMBIA', 'ECUADOR',
       'FRENCH POLYNESIA', 'NEW CALEDONIA', 'TURKS and CaICOS', 'CUBA',
       'BRAZIL', 'SEYCHELLES', 'ARGENTINA', 'FIJI', 'MeXICO', 'ENGLAND',
       'JAPAN', 'INDONESIA', 'JAMAICA', 'MALDIVES', 'THAILAND',
       'COLUMBIA', 'COSTA RICA', 'British Overseas Territory', 'CANADA',
       'JORDAN', 'ST KITTS / NEVIS', 'ST MARTIN', 'PAPUA

In [21]:
df['Country'].isnull().sum()

50

In [22]:
#unificando para que todo sea mayuscula o minusculas y quitando espacio en blanco 
df['Country'] = df['Country'].str.upper().str.strip()

In [ ]:
#Diccionario de reemplaszos para unificar variantes:
replacements = {
    'AUSTRALIA': 'AUSTRALIA',
    'USA': 'USA',
    'US VIRGIN ISLANDS': 'USA',
    'UNITED STATES': 'USA',
    'NEW ZEALAND': 'NEW ZEALAND',
    'FIJI': 'FIJI',
    'COLOMBIA': 'COLOMBIA',
    'COLUMBIA': 'COLOMBIA',
    'MEXICO': 'MEXICO',
    'MALDIVES': 'MALDIVES',
    'MALDIVE ISLANDS': 'MALDIVES',
    'TURKS AND CAICOS': 'TURKS & CAICOS',
    'TURKS & CAICOS': 'TURKS & CAICOS',
    'PORTUGAL': 'PORTUGAL',
    'SPAIN': 'SPAIN',
    'UK': 'UNITED KINGDOM',
    'UNITED KINGDOM': 'UNITED KINGDOM',
    'ENGLAND': 'UNITED KINGDOM',
    'FRANCE': 'FRANCE',
    'GERMANY': 'GERMANY',
    'ITALY': 'ITALY',
    'JAPAN': 'JAPAN',
    'INDONESIA': 'INDONESIA',
    'THAILAND': 'THAILAND',
    'PHILIPPINES': 'PHILIPPINES',
    'BRAZIL': 'BRAZIL',
    'ARGENTINA': 'ARGENTINA',
    'SOUTH AFRICA': 'SOUTH AFRICA',
    'SAUDI ARABIA': 'SAUDI ARABIA',
    'EGYPT': 'EGYPT',
    'ISRAEL': 'ISRAEL',
    'CANADA': 'CANADA',
    'CEYLON': 'SRI LANKA',
    'CEYLON (SRI LANKA)': 'SRI LANKA', 

    # Océanos y mares los podemos poner como "OCEAN"
    'MEDITERRANEAN SEA': 'OCEAN',
    'ATLANTIC OCEAN': 'OCEAN',
    'TASMAN SEA': 'OCEAN',
    'RED SEA?': 'OCEAN',
    'CORAL SEA': 'OCEAN',
    'INDIAN OCEAN?': 'OCEAN',
    'BETWEEN PORTUGAL & INDIA': 'OCEAN',
    
    # Continentes o zonas grandes
    'COAST OF AFRICA': 'AFRICA',
    
    # Territorios pequeños que se pueden dejar como están o agrupar
    'ST MARTIN': 'ST MARTIN',
    'ST. MARTIN': 'ST MARTIN',
    'TRINIDAD & TOBAGO': 'TRINIDAD & TOBAGO',

  
    # Nan o vacíos
    'BRITISH NEW GUINEA': 'UNKNOWN',
    'ANDAMAN ISLANDS' : 'UNKNOWN',
    'COOK ISLANDS': 'UNKNOWN',
    'ROATAN': 'UNKNOWN',
    None: 'UNKNOWN',
    '': 'UNKNOWN',
    'AFRICA' : 'REGION',
    'ASIA?' : 'REGION',
    'KOREA': 'UNKNOWN',
    'EQUATORIAL GUINEA / CAMEROON': 'UNKNOWN'
}


df['Country'] = df['Country'].replace(replacements)

### Hipotesis 2: Mueren mas hombres que mujeres?

In [24]:
df.head()

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source
0,10th January,2026.0,Unprovoked,AUSTRALIA,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,puncture mark to left thumb,N,0540hrs,Unknown,Bob Myatt GSAF
1,8th January,2026.0,Unprovoked,USA,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,Left arm torn off in the attack below the elbow,Y,1628hrs,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com
2,3rd January,2026.0,Unprovoked,NEW CALEDONIA,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,Injuries to upper limbs,N,?,Unknown,Andy Currie: Province Sud:
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,Taken by shark body recovered with multiple in...,Y,1200hrs,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,Hand Injury,N,0800hrs,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...


In [26]:
df['Sex'].unique()

array(['M', 'F', 'F ', 'M ', nan, ' M', 'm', 'lli', 'M x 2', 'N', '.'],
      dtype=object)

#### Eliminacion de Nulos de ['Sex']

In [20]:
data_sex = df[['Sex']]
data_sex.isnull().sum()

Sex    579
dtype: int64

In [21]:
sin_vacios = data_sex.dropna(subset=['Sex'])

In [22]:
print(data_sex.isnull().sum())

Sex    579
dtype: int64


data_sex.info()

In [23]:
sin_vacios.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6486 entries, 0 to 7064
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Sex     6486 non-null   object
dtypes: object(1)
memory usage: 101.3+ KB


In [24]:
sin_vacios['Sex'] = sin_vacios['Sex'].str.strip().str.upper()
sin_vacios['Sex'].unique()

/var/folders/m_/06pskt392pn6qvkq92krm8xm0000gn/T/ipykernel_66300/3308651882.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sin_vacios['Sex'] = sin_vacios['Sex'].str.strip().str.upper()


array(['M', 'F', 'LLI', 'M X 2', 'N', '.'], dtype=object)

In [25]:
#Unificar valores raros a 'Unkown
sin_vacios['Sex'] = sin_vacios['Sex'].replace({
    'M X 2': 'Unknown',
    'LLI': 'Unknown',
    'N': 'Unknown',
    '.': 'Unknown'       
})


/var/folders/m_/06pskt392pn6qvkq92krm8xm0000gn/T/ipykernel_66300/3112342755.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sin_vacios['Sex'] = sin_vacios['Sex'].replace({


In [26]:
sin_vacios['Sex'].unique()

array(['M', 'F', 'Unknown'], dtype=object)

In [27]:
sin_vacios.describe()

,Sex
count,6486
unique,3
top,M
freq,5671


In [28]:
sin_vacios.isnull().sum()

Sex    0
dtype: int64

#### Hipotesis 2: En que año hay mas ataques  

In [74]:
df['Year'].unique()

array([2026., 2025., 2024., 2023., 2022., 2021., 2020., 2019., 2018.,
       2017.,   nan, 2016., 2015., 2014., 2013., 2012., 2011., 2010.,
       2009., 2008., 2007., 2006., 2005., 2004., 2003., 2002., 2001.,
       2000., 1999., 1998., 1997., 1996., 1995., 1984., 1994., 1993.,
       1992., 1991., 1990., 1989., 1969., 1988., 1987., 1986., 1985.,
       1983., 1982., 1981., 1980., 1979., 1978., 1977., 1976., 1975.,
       1974., 1973., 1972., 1971., 1970., 1968., 1967., 1966., 1965.,
       1964., 1963., 1962., 1961., 1960., 1959., 1958., 1957., 1956.,
       1955., 1954., 1953., 1952., 1951., 1950., 1949., 1948., 1848.,
       1947., 1946., 1945., 1944., 1943., 1942., 1941., 1940., 1939.,
       1938., 1937., 1936., 1935., 1934., 1933., 1932., 1931., 1930.,
       1929., 1928., 1927., 1926., 1925., 1924., 1923., 1922., 1921.,
       1920., 1919., 1918., 1917., 1916., 1915., 1914., 1913., 1912.,
       1911., 1910., 1909., 1908., 1907., 1906., 1905., 1904., 1903.,
       1902., 1901.,

In [75]:
frequency_year = df['Year'].mode()[0]
frequency_year

2015.0

In [76]:
df[df['Year'] >= 1000]

,Date,Year,Type,Country,State,Location,Activity,Name,Sex,Age,Injury,Fatal Y/N,Time,Species,Source
0,10th January,2026.0,Unprovoked,Australia,NSW,Avalon Beach,Surfing,Paul Stanton,M,?,puncture mark to left thumb,N,0540hrs,Unknown,Bob Myatt GSAF
1,8th January,2026.0,Unprovoked,US Virgin Islands,Fredricksted Island St Croix,Dorsch Beach,Snorkeling,Arlene Lillis,F,56,Left arm torn off in the attack below the elbow,Y,1628hrs,Unknown,Todd Smith: KevinMcMurray Trackingsharks.com
2,3rd January,2026.0,Unprovoked,New Caledonia,Kélé,Between Bourail and Moindou,Scuba Diving,Unknown,M,?,Injuries to upper limbs,N,?,Unknown,Andy Currie: Province Sud:
3,21st December,2025.0,Unprovoked,USA,California,Lovers Point Pacific Grove,Swimming,Erica Fox,F,55,Taken by shark body recovered with multiple in...,Y,1200hrs,Great White Shark,Kevin McMurray Tracking sharks.com: Ralph Coll...
4,12th December,2025.0,Unprovoked,USA,Sonoma County California,Salmon Creek,Surfing,Unknown,M,?,Hand Injury,N,0800hrs,Suspected Great White Shark,Kevin McMurray Tracking sharks.com:Andrew Curr...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6929,Ca. 1554,1554.0,Unprovoked,FRANCE,Nice & Marseilles,NaN,NaN,males (wearing armor),M,NaN,NaN,UNKNOWN,NaN,Possibly white sharks,G. Rondelet
6930,Ca. 1543,1543.0,Unprovoked,VENEZUELA,Magarita or Cubagua Islands,NaN,Pearl diving,Indian slave,M,NaN,FATAL,Y,NaN,NaN,J. Castro
6931,Ca 1588.04.00,1518.0,Unprovoked,MEXICO,Yucatan,Cozumel,Swmming,A cacique (a chief),M,NaN,Toes severed,N,NaN,NaN,"C. Moore, GSAF"
6932,Ca 1200-1500 A.D.,1500.0,Unprovoked,MEXICO,NaN,NaN,NaN,male,M,NaN,Foot severed,N,NaN,NaN,J. Castro


In [77]:
df.loc[df['Year'] <= 1000, 'Year'] = np.nan

In [78]:
df['Year'].unique()

array([2026., 2025., 2024., 2023., 2022., 2021., 2020., 2019., 2018.,
       2017.,   nan, 2016., 2015., 2014., 2013., 2012., 2011., 2010.,
       2009., 2008., 2007., 2006., 2005., 2004., 2003., 2002., 2001.,
       2000., 1999., 1998., 1997., 1996., 1995., 1984., 1994., 1993.,
       1992., 1991., 1990., 1989., 1969., 1988., 1987., 1986., 1985.,
       1983., 1982., 1981., 1980., 1979., 1978., 1977., 1976., 1975.,
       1974., 1973., 1972., 1971., 1970., 1968., 1967., 1966., 1965.,
       1964., 1963., 1962., 1961., 1960., 1959., 1958., 1957., 1956.,
       1955., 1954., 1953., 1952., 1951., 1950., 1949., 1948., 1848.,
       1947., 1946., 1945., 1944., 1943., 1942., 1941., 1940., 1939.,
       1938., 1937., 1936., 1935., 1934., 1933., 1932., 1931., 1930.,
       1929., 1928., 1927., 1926., 1925., 1924., 1923., 1922., 1921.,
       1920., 1919., 1918., 1917., 1916., 1915., 1914., 1913., 1912.,
       1911., 1910., 1909., 1908., 1907., 1906., 1905., 1904., 1903.,
       1902., 1901.,

In [79]:
df = df.dropna(subset=['Year'])

In [80]:
df['Year'].isnull().sum()

0

In [ ]:
#df.shape()

TypeError: 'tuple' object is not callable